# 4th down: compute
Works through the data and runs all simulations, then saves everything to `results.json`.
The plots and the handbook are in **`results.ipynb`**, which only reads `results.json`.

`SPEED` scales the number of simulated games: 1.0 ≈ 30 min (final numbers), 0.2 ≈ 6 min (quick test).

In [1]:
import random
import numpy as np
import pandas as pd
import model as M

M.SPEED = 1.0
res = {"speed": M.SPEED}

## 1. Data and probabilities
nflverse play-by-play, regular season 2018–2025. Probabilities use only neutral situations (quarters 1–3).

In [2]:
pbp = M.load_data()
m = M.estimate(pbp)
print(f"{len(pbp):,} plays, {pbp.neutral.sum():,} in neutral situations")
print(f"Field goals: {m['n_fg']:,} kicks, logistic b0 = {m['fg_logit'][0]:.2f}, b1 = {m['fg_logit'][1]:.3f}")
print(f"Mean punt {np.mean([k for k, r in m['punts']]):.1f} yd, P(extra point) = {m['p_xp']:.3f}")
print(f"Drives per team per game: {m['drives_mean']:.2f} -> the simulator uses {m['drives']}")
print(f"Coaches go for it on {m['go_rate']:.1%} of {m['n_fourth']:,} 4th downs")
m["p_d"].T.round(3)

371,798 plays, 232,585 in neutral situations
Field goals: 8,241 kicks, logistic b0 = 6.18, b1 = -0.105
Mean punt 46.2 yd, P(extra point) = 0.944
Drives per team per game: 11.15 -> the simulator uses 11
Coaches go for it on 13.9% of 18,921 4th downs


ydstogo,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
mean,0.7,0.576,0.529,0.488,0.434,0.429,0.384,0.355,0.305,0.294,0.273,0.234,0.223,0.182,0.141
count,5641.0,3456.000,3178.000,3034.000,3092.000,2891.000,2731.000,2459.000,2079.000,2588.000,1132.000,931.000,754.000,625.000,638.000


## 2. Check the simulator
Coach vs coach should win 50%; simulated P(1st down) and how drives end should match the data.

In [3]:
out0, go0 = M.run(m, M.coach, n=50_000, seed=1)
p, lo, hi, pval = M.coach_vs_coach_test(out0)
print(f"Coach vs coach: p̂ = {p:.4f} [{lo:.4f}, {hi:.4f}], p-value = {pval:.2f}")
print(f"Simulated coaches go for it on {go0:.1%} of 4th downs (data {m['go_rate']:.1%})")

rng = random.Random(7)
def simulated_conversion(d):
    ys, p_to = m["yards"]["equal"][(3, d)]
    return np.mean([rng.random() >= p_to and ys[int(rng.random() * len(ys))] >= d for _ in range(20_000)])
conv = m["p_d"].loc[1:10].copy()
conv["simulated"] = [simulated_conversion(d) for d in conv.index]

ends = []
for i in range(5000):
    M.game(m, M.coach, "equal", random.Random(i), None, ends)
real = pbp.groupby(["game_id", "fixed_drive"]).fixed_drive_result.first()
real = real[~real.isin(["End of half", "Opp touchdown"])]
drive_ends = pd.DataFrame({"Data": real.value_counts(normalize=True),
                           "Simulator": pd.Series(ends).value_counts(normalize=True)}).fillna(0)
drive_ends = drive_ends.sort_values("Data", ascending=False)

res.update(coach_vs_coach=[p, lo, hi, pval], coach_go_sim=go0, conversion=conv, drive_ends=drive_ends)
display(conv.T.round(3))
drive_ends.round(3)

Coach vs coach: p̂ = 0.5022 [0.4979, 0.5065], p-value = 0.32
Simulated coaches go for it on 16.8% of 4th downs (data 13.9%)


ydstogo,1,2,3,4,5,6,7,8,9,10
mean,0.700,0.576,0.529,0.488,0.434,0.429,0.384,0.355,0.305,0.294
count,5641.000,3456.000,3178.000,3034.000,3092.000,2891.000,2731.000,2459.000,2079.000,2588.000
simulated,0.698,0.570,0.518,0.486,0.428,0.429,0.378,0.344,0.293,0.281


,Data,Simulator
Punt,0.391,0.426
Touchdown,0.245,0.246
Field goal,0.163,0.138
Turnover,0.112,0.110
Turnover on downs,0.057,0.045
Missed field goal,0.029,0.026
Safety,0.003,0.009


## 3. Step 1: each strategy on its own
### 3.1 How long a field goal should our team try?
Kick a field goal from up to K yards (otherwise punt), going for it on 4th & 4 or less. The best K is used from now on.

In [4]:
fg_sweep = {}
for K in [45, 50, 53, 55, 57, 59, 61, 63, 65, 68]:
    fg_sweep[K] = M.ci(M.run(m, M.strategy([M.short_yardage(4)], fg_range=K - 18), n=20_000, seed=10)[0])
fg_sweep = pd.DataFrame(fg_sweep, index=["p", "lo", "hi"]).T
best_k = int(fg_sweep.p.idxmax())
FG = best_k - 18                                    # our field-goal range in pos
print(f"Best K = {best_k} yards (pos ≤ {FG}), P(made) there = {m['p_fg'][FG]:.0%}")
res.update(fg_sweep=fg_sweep, best_k=best_k, fg_range=FG)

Best K = 61 yards (pos ≤ 43), P(made) there = 49%


### 3.2 Aggressiveness: go at random (strategy 4) vs go on short yardage (strategy 5)

In [5]:
def sweep(policies, seed):
    rows = []
    for key, pol in policies.items():
        out, go = M.run(m, pol, n=10_000, seed=seed)
        rows.append((key, go, *M.ci(out)))
    return pd.DataFrame(rows, columns=["value", "go", "p", "lo", "hi"])

sweep_q = sweep({q: M.fixed_rate(q, FG) for q in np.linspace(0, 1, 11)}, seed=11)
sweep_x = sweep({X: M.strategy([M.short_yardage(X)], FG) for X in [0, 1, 2, 3, 4, 5, 6, 8, 10, 15]}, seed=11)
out_c, go_c = M.run(m, M.coach, n=10_000, seed=11)
best_x = int(sweep_x.loc[sweep_x.p.idxmax(), "value"])
print(f"Best X = {best_x}")
res.update(sweep_q=sweep_q, sweep_x=sweep_x, coach_point=[go_c, out_c.mean()], best_x=best_x)
sweep_x.round(3)

Best X = 6


,value,go,p,lo,hi
0,0,0.000,0.467,0.458,0.477
1,1,0.140,0.502,0.493,0.512
2,2,0.232,0.523,0.513,0.532
3,3,0.322,0.532,0.522,0.541
4,4,0.408,0.537,0.527,0.547
5,5,0.494,0.543,0.534,0.553
6,6,0.569,0.545,0.535,0.554
7,8,0.706,0.542,0.532,0.551
8,10,0.821,0.516,0.506,0.526
9,15,0.927,0.492,0.482,0.502


### 3.3 Strategy 7: yards to go ≤ X and pos ≤ Y
The top cells are within noise of each other, so the best 3 are carried forward.

In [6]:
X_VALS, Y_VALS = [1, 2, 3, 4, 5, 6, 8, 10], [20, 30, 40, 50, 60, 70, 80, 100]
heat = pd.DataFrame(index=X_VALS, columns=Y_VALS, dtype=float)
for X in X_VALS:
    for Y in Y_VALS:
        heat.loc[X, Y] = M.run(m, M.strategy([M.short_and_close(X, Y)], FG), n=10_000, seed=12)[0].mean()
top_xy = [[int(x), int(y)] for x, y in heat.stack().sort_values(ascending=False).index[:3]]
print("Top 3 (X, Y):", top_xy)
res.update(heat=heat, top_xy=top_xy)

Top 3 (X, Y): [[5, 100], [6, 80], [6, 70]]


### 3.4 Strategy 10: trailing with ≤ N drives left

In [7]:
sweep_n = pd.Series({N: M.run(m, M.strategy([M.trailing_late(N)], FG), n=10_000, seed=13)[0].mean()
                     for N in [1, 2, 3, 4, 6, 8, 12]}, name="p̂")
best_n = int(sweep_n.idxmax())
res.update(sweep_n=sweep_n, best_n=best_n)
sweep_n.to_frame().T.round(3)

,1,2,3,4,6,8,12
p̂,0.483,0.49,0.496,0.497,0.497,0.498,0.504


### 3.5 Where is no man's land? (strategy 8)
On top of "go if ≤ X yards", also go whenever A ≤ pos ≤ B.

In [8]:
A_VALS, B_VALS = [25, 30, 35, 40, 45, 50], [35, 40, 45, 50, 55, 60, 65]
nml = pd.DataFrame(index=A_VALS, columns=B_VALS, dtype=float)
nml_base = M.run(m, M.strategy([M.short_yardage(best_x)], FG), n=10_000, seed=14)[0].mean()
for A in A_VALS:
    for B in B_VALS:
        if A < B:
            pol = M.strategy([M.short_yardage(best_x), M.no_mans_land(A, B)], FG)
            nml.loc[A, B] = M.run(m, pol, n=10_000, seed=14)[0].mean()
top_nml = [[int(a), int(b)] for a, b in nml.stack().sort_values(ascending=False).index[:3]]
print(f"Best no man's land: pos {top_nml[0]} ({100 * (nml.loc[top_nml[0][0], top_nml[0][1]] - nml_base):+.1f} pp). Top 3: {top_nml}")
res.update(nml=nml, nml_base=nml_base, top_nml=top_nml)

Best no man's land: pos [45, 50] (+0.0 pp). Top 3: [[45, 50], [30, 35], [35, 50]]


### 3.6 All single strategies (fresh random numbers, 30,000 paired games)

In [9]:
(X7, Y7), (A8, B8) = top_xy[0], top_nml[0]
singles = {
    "1. Real coaches": M.coach,
    "2. Never": M.strategy([], FG),
    "3. Always": M.always,
    "4. Fixed rate (q = 0.2)": M.fixed_rate(0.2, FG),
    f"5. Short yardage (X = {best_x})": M.strategy([M.short_yardage(best_x)], FG),
    "6. Past midfield": M.strategy([M.past_midfield()], FG),
    f"7. Short and close (X, Y = {X7}, {Y7})": M.strategy([M.short_and_close(X7, Y7)], FG),
    f"8. No man's land ({A8}–{B8})": M.strategy([M.no_mans_land(A8, B8)], FG),
    "9. Goal line": M.strategy([M.goal_line()], FG),
    f"10. Trailing late (N = {best_n})": M.strategy([M.trailing_late(best_n)], FG),
    "11. Score-aware FG, last 3 drives": M.strategy([], FG, fg11=True),
    "11. Score-aware FG on top of coaches": M.strategy([], FG, fg11=True, fallback="coach"),
}
singles_res = M.compare(m, singles, seed=100)
res["singles"] = singles_res
singles_res.round(3)

,p̂,diff (pp),diff lower,diff upper,go %
strategy,,,,,
1. Real coaches,0.500,0.000,0.000,0.000,16.806
2. Never,0.469,-3.133,-3.868,-2.399,0.000
3. Always,0.462,-3.840,-4.606,-3.074,100.000
4. Fixed rate (q = 0.2),0.467,-3.318,-4.075,-2.562,19.943
5. Short yardage (X = 6),0.538,3.788,3.032,4.545,56.681
6. Past midfield,0.509,0.880,0.124,1.636,47.310
"7. Short and close (X, Y = 5, 100)",0.537,3.650,2.900,4.400,49.164
8. No man's land (45–50),0.479,-2.192,-2.932,-1.452,7.905
9. Goal line,0.479,-2.180,-2.916,-1.444,2.947


## 4. Step 2: the best combination (successive halving)
Every candidate is a set of rules and their parameters. Play all on a few games, keep the best quarter,
play the survivors on 4× more (fresh) games, repeat.

In [10]:
cands = M.candidates(top_xy, top_nml)
print(f"{len(cands)} candidates")
history, games = M.successive_halving(m, cands, FG)
effects, r2 = M.rule_effects(cands, history[0])
final_ids = sorted(history[-1], key=history[-1].get, reverse=True)[:5]
res.update(halving=[sorted(h.values()) for h in history], halving_games=games, effects=effects, effects_r2=r2)
pd.DataFrame({"candidate": [M.label(cands[i]) for i in final_ids],
              "last-round diff (pp)": [history[-1][i] for i in final_ids]}).round(2)

2304 candidates


Round 1: 832 distinct candidates on 1,000 games each


Round 2: 208 distinct candidates on 4,000 games each


Round 3: 52 distinct candidates on 16,000 games each


Round 4: 13 distinct candidates on 64,000 games each


,candidate,last-round diff (pp)
0,≤4 yd + goal line + trailing (≤3 drives) | els...,4.57
1,≤4 yd + trailing (≤3 drives) | else kick,4.48
2,≤6 yd & pos ≤70 + trailing (≤2 drives) | else ...,4.33
3,≤4 yd + goal line + trailing (≤3 drives) + sco...,4.32
4,≤6 yd & pos ≤70 + trailing (≤3 drives) | else ...,4.32


### 4.1 Final check on fresh random numbers (100,000 paired games)

In [11]:
single_name = f"Best single rule: 5 (X = {best_x})"
final = {"1. Real coaches": M.coach, single_name: M.strategy([M.short_yardage(best_x)], FG)}
for i in final_ids:
    final[M.label(cands[i])] = M.build(cands[i], FG)
final_res = M.compare(m, final, n=100_000, seed=900)
best_name = final_res.drop(["1. Real coaches", single_name]).sort_values("p̂").index[-1]
best = cands[next(i for i in final_ids if M.label(cands[i]) == best_name)]

d = M.run(m, final[best_name], n=100_000, seed=900)[0] - M.run(m, final[single_name], n=100_000, seed=900)[0]
best_vs_single = M.ci(100 * d)
print(f"Best combination: {best_name}")
print("Best combination − best single rule = {:.2f} pp [{:.2f}, {:.2f}]".format(*best_vs_single))
res.update(final=final_res, best_name=best_name, best=best, best_vs_single=list(best_vs_single))
final_res.sort_values("p̂", ascending=False).round(3)

Best combination: ≤6 yd & pos ≤70 + trailing (≤2 drives) | else kick
Best combination − best single rule = 0.47 pp [0.24, 0.70]


,p̂,diff (pp),diff lower,diff upper,go %
strategy,,,,,
≤6 yd & pos ≤70 + trailing (≤2 drives) | else kick,0.544,4.492,4.081,4.903,54.592
≤4 yd + goal line + trailing (≤3 drives) | else kick,0.544,4.466,4.055,4.876,49.350
≤4 yd + trailing (≤3 drives) | else kick,0.544,4.418,4.008,4.828,49.014
≤6 yd & pos ≤70 + trailing (≤3 drives) | else kick,0.543,4.341,3.930,4.752,56.833
≤4 yd + goal line + trailing (≤3 drives) + score-aware FG | else kick,0.543,4.312,3.902,4.722,51.946
Best single rule: 5 (X = 6),0.540,4.022,3.610,4.434,56.817
1. Real coaches,0.500,0.000,0.000,0.000,16.774


## 5. Step 3: opponent strength
Strong / weak opponent = plays only from the top / bottom third of teams by EPA per play.

In [12]:
OPPS = ["weak", "equal", "strong"]
X_OPP = [0, 1, 2, 3, 4, 6, 8, 10, 15]                       # X = 0 is "never"
aggr = pd.DataFrame(index=[str(x) for x in X_OPP] + ["always"], columns=OPPS, dtype=float)
for k, o in enumerate(OPPS):
    base = M.run(m, M.coach, o, n=10_000, seed=1000 + k)[0]
    for X in X_OPP:
        aggr.loc[str(X), o] = 100 * (M.run(m, M.strategy([M.short_yardage(X)], FG), o, n=10_000, seed=1000 + k)[0] - base).mean()
    aggr.loc["always", o] = 100 * (M.run(m, M.always, o, n=10_000, seed=1000 + k)[0] - base).mean()
best_x_opp = {o: aggr[o].drop("always").idxmax() for o in OPPS}
print("Best X per opponent:", best_x_opp)

opp_res = pd.DataFrame({o: M.compare(m, {f"best single (≤{best_x} yd)": final[single_name],
                                         "best combination": final[best_name]}, o, n=30_000, seed=1100 + k)["diff (pp)"]
                        for k, o in enumerate(OPPS)})
res.update(aggr=aggr, best_x_opp=best_x_opp, opp_res=opp_res)
opp_res.round(1)

Best X per opponent: {'weak': '4', 'equal': '4', 'strong': '6'}


,weak,equal,strong
strategy,,,
best single (≤6 yd),2.7,4.0,5.1
best combination,3.5,4.2,5.3


## 6. Save

In [13]:
M.save_results(res, "results.json")
print("Saved results.json")

Saved results.json
